# Notebook 1 : Initialisation et Détection d'Objets

## Objectif
Ce premier notebook a pour but de mettre en place l'environnement de développement et d'effectuer les premières inférences sur le flux vidéo. L'objectif est de valider la capacité du modèle à identifier correctement les joueurs, les arbitres et le ballon avant de passer aux étapes de traitement plus complexes.


## 1. Installation des dépendances

In [1]:
#!pip install ultralytics opencv-python numpy pandas matplotlib yt_dlp
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt


## 2. Chargement du modèle

In [10]:
# Chargement du modèle pré-entraîné sur le dataset COCO
model = YOLO('yolov8m.pt') 
# Test rapide sur une image aléatoire du web pour voir si tout fonctionne
results = model('https://ultralytics.com/images/bus.jpg')


image 1/1 /Users/arnaudgrassian/Downloads/xg/bus.jpg: 640x480 4 persons, 1 bus, 165.4ms
Speed: 6.6ms preprocess, 165.4ms inference, 4.9ms postprocess per image at shape (1, 3, 640, 480)


## 3. Configuration du flux vidéo
Nous ouvrons la vidéo et récupérons ses propriétés (FPS, Largeur, Hauteur) pour préparer l'enregistrement du résultat.

In [11]:
video_path = "input_video.mp4"
cap = cv2.VideoCapture(video_path)

# Propriétés de la vidéo
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Configuration de l'écriture vidéo (codec mp4v)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_detection.mp4', fourcc, fps, (width, height))

print(f"Vidéo chargée : {width}x{height} à {fps} FPS")

Vidéo chargée : 1920x1080 à 25 FPS


## 4. Boucle de détection
Nous lisons la vidéo image par image. Pour chaque image, YOLO détecte les objets. Nous filtrons pour ne garder que la classe `0` (Personne) et `32` (Ballon de sport).

In [12]:
# Lire seulement les 20 premières secondes pour le test (fps * 20)
max_frames = fps * 20 
frame_count = 0

while cap.isOpened() and frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    # conf=0.3 : on ne garde que les détections sûres à >30%
    results = model(frame, conf=0.3, classes=[0, 32], verbose=False)
    
    # Annoter la frame avec les résultats (dessine les boîtes)
    annotated_frame = results[0].plot()
    
    # Sauvegarder la frame annotée
    out.write(annotated_frame)
    frame_count += 1
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"{frame_count} frames analysées. Vidéo sauvegardée sous 'output_detection.mp4'")

500 frames analysées. Vidéo sauvegardée sous 'output_detection.mp4'
